In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "Bangladesh Legal Acts Dataset.zip":
            print(os.path.join(root, file))

/content/drive/MyDrive/Bangladesh Legal Acts Dataset.zip


In [3]:
zip_path = "/content/drive/MyDrive/Bangladesh Legal Acts Dataset.zip"

In [4]:
import zipfile
import os

extract_dir = "/content/legal_dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted to:", extract_dir)

Dataset extracted to: /content/legal_dataset


In [5]:
import os

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(os.path.join(root, file))

/content/legal_dataset/govt.json
/content/legal_dataset/bangladesh_legal_systems.json
/content/legal_dataset/Contextualized_Bangladesh_Legal_Acts.json
/content/legal_dataset/balanced_1000.csv
/content/legal_dataset/filtered_act_list.csv
/content/legal_dataset/acts/act-print-448.json
/content/legal_dataset/acts/act-print-345.json
/content/legal_dataset/acts/act-print-1370.json
/content/legal_dataset/acts/act-print-61.json
/content/legal_dataset/acts/act-print-1135.json
/content/legal_dataset/acts/act-print-87.json
/content/legal_dataset/acts/act-print-581.json
/content/legal_dataset/acts/act-print-502.json
/content/legal_dataset/acts/act-print-1361.json
/content/legal_dataset/acts/act-print-283.json
/content/legal_dataset/acts/act-print-207.json
/content/legal_dataset/acts/act-print-1116.json
/content/legal_dataset/acts/act-print-120.json
/content/legal_dataset/acts/act-print-956.json
/content/legal_dataset/acts/act-print-664.json
/content/legal_dataset/acts/act-print-1158.json
/content

In [6]:
import os
import json
import pandas as pd
from pathlib import Path

In [7]:
json_files = []

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".json"):
            json_files.append(os.path.join(root, file))

print(f"Total JSON files found: {len(json_files)}")

# Show the first 10 files
for f in json_files[:10]:
    print(f)

Total JSON files found: 1487
/content/legal_dataset/govt.json
/content/legal_dataset/bangladesh_legal_systems.json
/content/legal_dataset/Contextualized_Bangladesh_Legal_Acts.json
/content/legal_dataset/acts/act-print-448.json
/content/legal_dataset/acts/act-print-345.json
/content/legal_dataset/acts/act-print-1370.json
/content/legal_dataset/acts/act-print-61.json
/content/legal_dataset/acts/act-print-1135.json
/content/legal_dataset/acts/act-print-87.json
/content/legal_dataset/acts/act-print-581.json


In [8]:
loaded_jsons = []

for file_path in json_files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            loaded_jsons.append({
                "file_path": file_path,
                "data": data
            })
    except Exception as e:
        print(f"Could not read {file_path}: {e}")

print(f"Successfully loaded {len(loaded_jsons)} JSON files.")

Successfully loaded 1487 JSON files.


In [9]:
first = loaded_jsons[0]

print("File:")
print(first["file_path"])

print("\nTop-level type:")
print(type(first["data"]))

if isinstance(first["data"], dict):
    print("\nTop-level keys:")
    print(list(first["data"].keys()))
elif isinstance(first["data"], list):
    print(f"\nThis JSON contains a list with {len(first['data'])} items.")

File:
/content/legal_dataset/govt.json

Top-level type:
<class 'list'>

This JSON contains a list with 52 items.


In [10]:
def extract_text(obj):
    """
    Recursively extract all text values from nested JSON.
    """
    texts = []

    if isinstance(obj, dict):
        for value in obj.values():
            texts.extend(extract_text(value))

    elif isinstance(obj, list):
        for item in obj:
            texts.extend(extract_text(item))

    elif isinstance(obj, str):
        text = obj.strip()
        if len(text) > 0:
            texts.append(text)

    return texts

In [11]:
documents = []

for item in loaded_jsons:
    file_path = item["file_path"]
    data = item["data"]

    texts = extract_text(data)
    full_text = " ".join(texts)

    documents.append({
        "file": os.path.basename(file_path),
        "text": full_text
    })

print("Total documents:", len(documents))

Total documents: 1487


In [12]:
df = pd.DataFrame(documents)

print(df.head())

print("\nShape:")
print(df.shape)

                                        file  \
0                                  govt.json   
1              bangladesh_legal_systems.json   
2  Contextualized_Bangladesh_Legal_Acts.json   
3                         act-print-448.json   
4                         act-print-345.json   

                                                text  
0  Company Rule (Dual System until 1772) Governor...  
1  Complex Legal and Governmental Systems Analysi...  
2  Contextualized Bangladesh Legal Acts Dataset C...  
3  The Bangladesh Laws (Repealing and Amending) O...  
4  The Ex-Government Servants (Employment with Fo...  

Shape:
(1487, 2)


In [13]:
df = df.drop_duplicates(subset=["text"])
df = df[df["text"].str.len() > 100]

df = df.reset_index(drop=True)

print("Remaining documents:", len(df))

Remaining documents: 1487


In [14]:
print(df.iloc[0]["file"])

print("\n================ SAMPLE TEXT ================\n")

print(df.iloc[0]["text"][:3000])

govt.json

================ SAMPLE TEXT ================

Company Rule (Dual System until 1772) Governor-General of Bengal Marquess of Wellesley (Richard Wellesley) Governor-General of Fort William in Bengal East India Company appointment Company Rule Governor-General of Bengal Marquess Cornwallis / Sir George Barlow (acting) Governor-General of Fort William in Bengal Company appointment Company Rule Governor-General of Bengal Lord Minto (Gilbert Elliot-Murray-Kynynmound) Governor-General of Fort William in Bengal Company appointment Company Rule Governor-General of India Marquess of Hastings (Francis Rawdon-Hastings) Governor-General of India (ex-officio Governor of Bengal) Company appointment Company Rule Governor-General of India Lord Amherst (William Amherst) Governor-General of India Company appointment Company Rule Governor-General of India Lord William Bentinck Governor-General of India Company appointment Company Rule Governor-General of India Lord Auckland (George Eden) Govern

In [15]:
output_path = "/content/legal_dataset_clean.csv"

df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/legal_dataset_clean.csv


In [16]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 29.4 MB/s eta 0:00:00


In [17]:
from sentence_transformers import SentenceTransformer

# Load a strong pretrained SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded successfully!


In [18]:
texts = df["text"].astype(str).tolist()

print("Number of documents:", len(texts))
print("First document preview:")
print(texts[0][:500])

Number of documents: 1487
First document preview:
Company Rule (Dual System until 1772) Governor-General of Bengal Marquess of Wellesley (Richard Wellesley) Governor-General of Fort William in Bengal East India Company appointment Company Rule Governor-General of Bengal Marquess Cornwallis / Sir George Barlow (acting) Governor-General of Fort William in Bengal Company appointment Company Rule Governor-General of Bengal Lord Minto (Gilbert Elliot-Murray-Kynynmound) Governor-General of Fort William in Bengal Company appointment Company Rule Gover


In [19]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embedding shape: (1487, 384)


In [20]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Number of indexed passages:", index.ntotal)

Number of indexed passages: 1487


In [21]:
query = "What is the law about property transfer?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

k = 5
scores, indices = index.search(query_embedding, k)

for rank, idx in enumerate(indices[0], start=1):
    print("=" * 80)
    print(f"Rank {rank}")
    print(df.iloc[idx]["file"])
    print(df.iloc[idx]["text"][:1000])

Rank 1
act-print-48.json
1The Transfer of Property Act, 1882 IV 1882 19/07/2025 PRELIMINARY 1.  This Act may be called theTransfer of Property Act, 1882. It shall come into force on the first day of July, 1882. 3[It extends to the whole of Bangladesh.] 2.4[Nothing herein contained shall be deemed to affect]-(a) 	the provisions of any enactment not hereby expressly repealed:(b) 	any terms or incidents of any contract or constitution of property which are consistent with the provisions of this Act, and are allowed by the law for the time being in force:(c) 	any right or liability arising out of a legal relation constituted before this Act comes into force, or any relief in respect of any such right or liability: or(d) 	save as provided by section 57 and Chapter IV of this Act, any transfer by operation of law or by, or in execution of, a decree or order of a Court of competent jurisdiction:and nothing in the second chapter of this Act shall be deemed to affect any rule of5[Muslim] law. 3

In [22]:
import os
import faiss

save_dir = "/content/legal_artifacts"
os.makedirs(save_dir, exist_ok=True)

faiss.write_index(index, os.path.join(save_dir, "legal_faiss.index"))

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [23]:
import numpy as np

np.save(
    os.path.join(save_dir, "legal_embeddings.npy"),
    embeddings
)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [24]:
df.to_csv(
    os.path.join(save_dir, "legal_documents.csv"),
    index=False
)

print("Document table saved successfully!")

Document table saved successfully!


In [25]:
test_queries = [
    "What is the law regarding marriage?",
    "How does property transfer work?",
    "What are the rules about taxation?",
    "What is cyber crime?",
    "What are labour rights?"
]

for query in test_queries:

    print("=" * 100)
    print("QUESTION:", query)

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, 3)

    for rank, idx in enumerate(indices[0], start=1):
        print(f"\nTop {rank} Match")
        print(df.iloc[idx]["file"])
        print(df.iloc[idx]["text"][:500])

QUESTION: What is the law regarding marriage?

Top 1 Match
act-print-25.json
1The Special Marriage Act, 1872 III 1872 19/07/2025 1. This Act extends to the whole of3[Bangladesh]. 2. Marriages may be celebrated under this Act between persons neither of whom professes the Christian or the Jewish, or the Hindu or the Muslim or the Parsi or the Buddhist, or the Sikh or the Jaina religion, or between persons each of whom professes one or other of the following religions, that is to say, the Hindu, Buddhist, Sikh or Jaina religion upon the following conditions:–(1)	neither par

Top 2 Match
act-print-83.json
The Foreign Marriage Act, 1903 XIV 1903 19/07/2025 1.(1) This Act may be called theForeign Marriage Act, 1903.(2) It extends to the whole of1[Bangladesh].(3) It applies also to all citizens of2[Bangladesh] and to all persons in the service of Government, whether citizens of3[Bangladesh] or not4[* * *]. 2.(1) Notice in writing of a marriage which it is intended to solemnize under the Forei

In [26]:
model_save_path = os.path.join(save_dir, "sbert_model")

model.save(model_save_path)

print("SBERT model saved to:", model_save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT model saved to: /content/legal_artifacts/sbert_model


In [27]:
from sentence_transformers import SentenceTransformer
import faiss

loaded_model = SentenceTransformer(model_save_path)

loaded_index = faiss.read_index(
    os.path.join(save_dir, "legal_faiss.index")
)

print("Reload successful!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Reload successful!


In [28]:
def search_legal_documents(query, top_k=5):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "file": df.iloc[idx]["file"],
            "text": df.iloc[idx]["text"]
        })

    return results

In [29]:
query = "What is the punishment for corruption?"

results = search_legal_documents(query, top_k=3)

for i, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("Similarity Score:", item["score"])
    print("Source File:", item["file"])
    print("\nPreview:\n")
    print(item["text"][:1000])

Result 1
Similarity Score: 0.5411316752433777
Source File: act-print-217.json

Preview:

The Prevention of Corruption Act, 1947 II 1947 19/07/2025 1. (1) This Act may be called thePrevention of Corruption Act, 1947.(2) It extends to the whole of Bangladesh and applies to all citizens of Bangladesh and persons in the service of1[the Republic] wherever they may be. 2. For the purposes of this Act, “public servant” means a public servant as defined in section 21 of the Penal Code and includes an employee of any corporation or other body or organisation set by the Government and includes a Chairman, Vice-Chairman, Member, Officer or other employee of a local2[authority], or a Chairman, Director, Managing Director, Trustee, Member, Officer or other employee of any corporation, or other body or organisation constituted or established under any law. 3. An offence punishable under section 161, 162, 163, 164, 165 or 165-A of the Penal Code shall be deemed to be a cognizable offence for the purp

In [30]:
import shutil
import os

drive_output = "/content/drive/MyDrive/Bangladesh_Legal_Project/artifacts"
os.makedirs(drive_output, exist_ok=True)

shutil.copy(
    "/content/legal_artifacts/legal_faiss.index",
    os.path.join(drive_output, "legal_faiss.index")
)

shutil.copy(
    "/content/legal_artifacts/legal_embeddings.npy",
    os.path.join(drive_output, "legal_embeddings.npy")
)

shutil.copy(
    "/content/legal_artifacts/legal_documents.csv",
    os.path.join(drive_output, "legal_documents.csv")
)

print("Artifacts copied to Google Drive.")

Artifacts copied to Google Drive.


In [31]:
def legal_search(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = {
            "rank": len(results) + 1,
            "score": float(score),
            "source_file": df.iloc[idx]["file"],
            "content": df.iloc[idx]["text"][:3000]
        }

        results.append(result)

    return results

In [32]:
question = "What are the rules regarding inheritance in Bangladesh?"

results = legal_search(question, top_k=5)

for item in results:

    print("="*100)

    print("Rank:", item["rank"])
    print("Score:", item["score"])
    print("Source:", item["source_file"])

    print("\nContent Preview:\n")

    print(item["content"])

Rank: 1
Score: 0.6873713731765747
Source: act-print-148.json

Content Preview:

The Hindu Law of Inheritance (Amendment) Act, 1929 II 1929 19/07/2025 1. (1) This Act may be called theHindu Law of Inheritance (Amendment) Act, 1929.(2) It extends to the whole of  [Bangladesh], but it applies only to persons who, but for the passing of this Act, would have been subject to the law of Mitakshara in respect of the provision herein enacted, and it applies to such persons in respect only of the property of males not held in coparcenary and not disposed of by will. 2. A son's daughter, daughter's daughter, sister, and sister's son shall, in the order so specified, be entitled to rank in the order of succession next after a father's father and before a father's brother:Provided that a sister's son shall not include a son adopted after the sister's death. 3. Nothing in this Act shall-(a) 	affect any special family or local custom having the force of law, or(b) 	vest in a son's daughter, daughter'

In [33]:
import json

question = "What is the punishment for cyber crime?"

results = legal_search(question)

response = {
    "question": question,
    "results": results
}

print(json.dumps(response, indent=4))

{
    "question": "What is the punishment for cyber crime?",
    "results": [
        {
            "rank": 1,
            "score": 0.4412204623222351,
            "source_file": "act-print-91.json",
            "content": "The Whipping Act, 1909 IV 1909 19/07/2025 1. (1) This Act may be called theWhipping Act, 1909; and(2)  It extends to the whole of Bangladesh. 2. In addition to the punishments described in section 53 of the Penal Code, offenders are also liable to the punishment of whipping. 3. Whoever commits any of the following offences, namely:-(a)\ttheft, as defined in section 378 of the Penal Code other than theft by a clerk or servant of property in possession of his master;(b)\ttheft in a building, tent or vessel, as defined in section 380 of the said Code;(c)\ttheft after preparation for causing death or hurt, as defined in section 382 of the said Code;(d)\tlurking house-trespass, or house-breaking, as defined in sections 443 and 445 of the said Code, in order to the commit

In [34]:
output_json = "/content/legal_search_output.json"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(response, f, indent=4, ensure_ascii=False)

print("Saved:", output_json)

Saved: /content/legal_search_output.json


In [35]:
def legal_qa(question):

    results = legal_search(question, top_k=3)

    answer = ""

    for r in results:
        answer += r["content"]
        answer += "\n\n"

    return answer[:5000]

In [36]:
question = "What are labour rights in Bangladesh?"

answer = legal_qa(question)

print(answer)

The Bidi (Restriction of Manufacture) (Repeal) Act, 1974 IX 1974 19/07/2025 1. (1) This Act may be called theBidi (Restriction of Manufacture) (Repeal) Act, 1974.(2) It shall come into force at once. 2. The Bidi (Restriction of Manufacture) Ordinance, 1966 (E.P. Ord. IV of 1966) is hereby repealed. Copyright©2019, Legislative and Parliamentary Affairs DivisionMinistry of Law, Justice and Parliamentary Affairs http://bdlaws.minlaw.gov.bd/act-print-459.html 2025-07-19 02:15:59 The Bidi (Restriction of Manufacture) (Repeal) Act, 1974 IX 1974 english Parliamentary Democracy Prime Minister Sheikh Mujibur Rahman Prime Minister of Bangladesh Democratic election (1973) 1973-1975 2025-07-19 19:33:21 english 2025-07-19 20:26:27 2025-07-19 20:36:05 2025-07-19 20:56:10 Early Independent Bangladesh 1971-1975 Laws Continuance Enforcement Order 1971 Constitution of Bangladesh 1972 Supreme Court with High Court and Appellate Divisions Subordinate Courts Secular constitutional democracy with British-in

In [37]:
import shutil
import os

drive_dir = "/content/drive/MyDrive/Bangladesh_Legal_Project/final_artifacts"

os.makedirs(drive_dir, exist_ok=True)

files_to_copy = [
    "/content/legal_artifacts/legal_faiss.index",
    "/content/legal_artifacts/legal_embeddings.npy",
    "/content/legal_artifacts/legal_documents.csv",
    "/content/legal_search_output.json"
]

for file_path in files_to_copy:

    if os.path.exists(file_path):

        shutil.copy(
            file_path,
            os.path.join(
                drive_dir,
                os.path.basename(file_path)
            )
        )

print("All artifacts copied to Google Drive.")

All artifacts copied to Google Drive.


In [38]:
import os

drive_model_path = "/content/drive/MyDrive/Bangladesh_Legal_Project/models/sbert_model"

os.makedirs(
    "/content/drive/MyDrive/Bangladesh_Legal_Project/models",
    exist_ok=True
)

model.save(drive_model_path)

print("SBERT model saved to:")
print(drive_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT model saved to:
/content/drive/MyDrive/Bangladesh_Legal_Project/models/sbert_model


In [39]:
import faiss

drive_index_path = "/content/drive/MyDrive/Bangladesh_Legal_Project/models/legal_faiss.index"

faiss.write_index(index, drive_index_path)

print("FAISS index saved to:")
print(drive_index_path)

FAISS index saved to:
/content/drive/MyDrive/Bangladesh_Legal_Project/models/legal_faiss.index


In [40]:
drive_csv_path = "/content/drive/MyDrive/Bangladesh_Legal_Project/models/legal_documents.csv"

df.to_csv(
    drive_csv_path,
    index=False
)

print("Document CSV saved to:")
print(drive_csv_path)

Document CSV saved to:
/content/drive/MyDrive/Bangladesh_Legal_Project/models/legal_documents.csv


In [41]:
def ask_legal_question(question, top_k=5):

    embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, ids = index.search(embedding, top_k)

    answers = []

    for score, idx in zip(scores[0], ids[0]):

        answers.append({
            "score": float(score),
            "source": df.iloc[idx]["file"],
            "text": df.iloc[idx]["text"]
        })

    return answers

In [42]:
while True:

    user_question = input("Ask a legal question (type exit to stop): ")

    if user_question.lower() == "exit":
        break

    results = ask_legal_question(user_question)

    print("\n")

    for i, result in enumerate(results, start=1):

        print("=" * 80)
        print(f"Result {i}")
        print("Similarity:", result["score"])
        print("Source:", result["source"])
        print()
        print(result["text"][:1000])
        print()

Ask a legal question (type exit to stop): What is the punishment for theft in Bangladesh?


Result 1
Similarity: 0.6279245615005493
Source: act-print-91.json

The Whipping Act, 1909 IV 1909 19/07/2025 1. (1) This Act may be called theWhipping Act, 1909; and(2)  It extends to the whole of Bangladesh. 2. In addition to the punishments described in section 53 of the Penal Code, offenders are also liable to the punishment of whipping. 3. Whoever commits any of the following offences, namely:-(a)	theft, as defined in section 378 of the Penal Code other than theft by a clerk or servant of property in possession of his master;(b)	theft in a building, tent or vessel, as defined in section 380 of the said Code;(c)	theft after preparation for causing death or hurt, as defined in section 382 of the said Code;(d)	lurking house-trespass, or house-breaking, as defined in sections 443 and 445 of the said Code, in order to the committing of any offence punishable with whipping under this section;(e)	l

In [43]:
sample_questions = [
    "What are labour rights?",
    "What is inheritance law?",
    "What is contract law?",
    "What are tax rules?",
    "What is cyber crime?"
]

for q in sample_questions:

    print("\n")
    print("#" * 100)
    print("QUESTION:", q)

    answers = ask_legal_question(q, top_k=1)

    print("TOP MATCH:")
    print(answers[0]["source"])
    print()
    print(answers[0]["text"][:800])



####################################################################################################
QUESTION: What are labour rights?
TOP MATCH:
act-print-666.json

The Agricultural Labour (Minimum Wages) Ordinance, 1984 XVII 1984 19/07/2025 1. This Ordinance may be called theAgricultural Labour (Minimum Wages) Ordinance, 1984. 2. In this Ordinance, unless there is anything repugnant in the subject or context,-(a)	“agricultural labourer” means any person employed in agricultural crop  production, but does not include-(i)	a person employed by the Government;(ii)	a person employed in a plantation as defined in clause (iii) of section 2 of the Payment of Wages Act, 1936 (IV of 1936);(iii)	a person who works as a family labourer on monthly wages;(iv)	a person employed by a company registered under the Companies Act, 1913 (VII of 1913), engaged in production and sale of fish or livestock of any kind;(v)	a bargadar as defined in theLand Reforms Ordinance, 1


#############################

In [44]:
import json
import os

metadata = {
    "model_name": "all-MiniLM-L6-v2",
    "num_documents": len(df),
    "embedding_dimension": int(embeddings.shape[1]),
    "faiss_index_type": "IndexFlatIP"
}

metadata_path = "/content/legal_artifacts/retrieval_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved:", metadata_path)

Saved: /content/legal_artifacts/retrieval_metadata.json


In [45]:
def search_with_metadata(question, top_k=5):

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    output = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):

        output.append({
            "rank": rank,
            "similarity_score": float(score),
            "document_name": df.iloc[idx]["file"],
            "document_preview": df.iloc[idx]["text"][:1000]
        })

    return output

In [46]:
questions = [
    "What are labour laws in Bangladesh?",
    "How is inheritance regulated?",
    "What is contract law?",
    "How are taxes imposed?",
    "What is the punishment for fraud?"
]

evaluation_results = {}

for q in questions:

    evaluation_results[q] = search_with_metadata(
        q,
        top_k=3
    )

print("Evaluation completed.")

Evaluation completed.


In [47]:
evaluation_path = "/content/legal_artifacts/evaluation_results.json"

with open(evaluation_path, "w", encoding="utf-8") as f:

    json.dump(
        evaluation_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved:", evaluation_path)

Saved: /content/legal_artifacts/evaluation_results.json


In [48]:
for question, answers in evaluation_results.items():

    print("=" * 100)
    print("QUESTION:")
    print(question)

    for answer in answers:

        print("-" * 80)
        print("Rank:", answer["rank"])
        print("Similarity:", answer["similarity_score"])
        print("Document:", answer["document_name"])
        print()
        print(answer["document_preview"][:500])
        print()

QUESTION:
What are labour laws in Bangladesh?
--------------------------------------------------------------------------------
Rank: 1
Similarity: 0.6284351348876953
Document: act-print-361.json

The National Service Ordinance, 1970 XXII 1970 19/07/2025 1. (1) This Ordinance may be called the2[* * *]National Service Ordinance, 1970.(2) It extends to the whole of Bangladesh and applies to all citizens of Bangladesh wherever they may be.(3) It shall come into force on such day as the Government may, by notification in the official Gazette, appoint in this behalf. 2. In this Ordinance, unless there is anything repugnant in the subject or context,-(a)	“emoluments”, in relation to a person, i

--------------------------------------------------------------------------------
Rank: 2
Similarity: 0.6262510418891907
Document: act-print-391.json

The Bangladesh Nationalised Enterprises and Statutory Corporation (Prohibition of Strikes and Unfair Labour Practices) Order, 1972 (President's Order) 5

In [49]:
def pretty_search(question, top_k=5):
    results = search_legal_documents(question, top_k=top_k)

    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    for i, r in enumerate(results, start=1):
        print(f"\nRank: {i}")
        print(f"Similarity Score: {r['score']:.4f}")
        print(f"Source File: {r['file']}")
        print("-" * 80)
        print(r['text'][:1500])
        print()

In [50]:
sample_questions = [
    "What are labour laws in Bangladesh?",
    "What is inheritance law?",
    "What is the punishment for corruption?",
    "How are taxes collected?",
    "What are contract obligations?"
]

for q in sample_questions:
    pretty_search(q, top_k=3)

QUESTION:
What are labour laws in Bangladesh?

Rank: 1
Similarity Score: 0.6284
Source File: act-print-361.json
--------------------------------------------------------------------------------
The National Service Ordinance, 1970 XXII 1970 19/07/2025 1. (1) This Ordinance may be called the2[* * *]National Service Ordinance, 1970.(2) It extends to the whole of Bangladesh and applies to all citizens of Bangladesh wherever they may be.(3) It shall come into force on such day as the Government may, by notification in the official Gazette, appoint in this behalf. 2. In this Ordinance, unless there is anything repugnant in the subject or context,-(a)	“emoluments”, in relation to a person, includes the pay, allowances, gratuity, fees, commissions, perquisites and profits which such person is entitled to receive from his employer for his services;(b)	“employer” means any person or body of persons, whether incorporated or not, who or which employs another person for hire or reward, either direc

In [51]:
import os

output_dir = "/content/drive/MyDrive/Bangladesh_Legal_Project/final_output"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "legal_documents.csv")
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)

Saved: /content/drive/MyDrive/Bangladesh_Legal_Project/final_output/legal_documents.csv


In [52]:
summary = f"""
Bangladesh Legal AI Project Summary

Total documents: {len(df)}
Embedding dimension: {embeddings.shape[1]}
Indexed passages: {index.ntotal}
SBERT model: all-MiniLM-L6-v2
"""

summary_path = os.path.join(output_dir, "project_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)
print("Saved:", summary_path)


Bangladesh Legal AI Project Summary

Total documents: 1487
Embedding dimension: 384
Indexed passages: 1487
SBERT model: all-MiniLM-L6-v2

Saved: /content/drive/MyDrive/Bangladesh_Legal_Project/final_output/project_summary.txt


In [53]:
while True:
    question = input("\nAsk a legal question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Exiting...")
        break

    results = search_legal_documents(question, top_k=3)

    print("\nTop Results:\n")

    for i, result in enumerate(results, start=1):
        print("=" * 80)
        print(f"Result {i}")
        print(f"Rank: {i}") # Using 'i' for rank
        print(f"Score: {result['score']:.4f}")
        print(f"Source File: {result['file']}") # Changed to 'file'
        print("\nRelevant Passage:\n")
        print(result['text'][:1500]) # Changed to 'text'
        print()


Ask a legal question (type 'exit' to quit): What is the punishment for theft in Bangladesh?

Top Results:

Result 1
Rank: 1
Score: 0.6279
Source File: act-print-91.json

Relevant Passage:

The Whipping Act, 1909 IV 1909 19/07/2025 1. (1) This Act may be called theWhipping Act, 1909; and(2)  It extends to the whole of Bangladesh. 2. In addition to the punishments described in section 53 of the Penal Code, offenders are also liable to the punishment of whipping. 3. Whoever commits any of the following offences, namely:-(a)	theft, as defined in section 378 of the Penal Code other than theft by a clerk or servant of property in possession of his master;(b)	theft in a building, tent or vessel, as defined in section 380 of the said Code;(c)	theft after preparation for causing death or hurt, as defined in section 382 of the said Code;(d)	lurking house-trespass, or house-breaking, as defined in sections 443 and 445 of the said Code, in order to the committing of any offence punishable with wh

In [54]:
while True:
    question = input("\nAsk a legal question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Exiting...")
        break

    results = search_legal_documents(question, top_k=3)

    print("\nTop Results:\n")

    for i, result in enumerate(results, start=1):
        print("=" * 80)
        print(f"Result {i}")
        print(f"Rank: {i}") # Corrected to use 'i' for rank
        print(f"Score: {result['score']:.4f}")
        print(f"Source File: {result['file']}") # Corrected to use 'file'
        print("\nRelevant Passage:\n")
        print(result['text'][:1500]) # Corrected to use 'text'
        print()


Ask a legal question (type 'exit' to quit): exit
Exiting...


In [57]:
while True:
    question = input("\nAsk a legal question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Exiting...")
        break

    results = search_legal_documents(question, top_k=3)

    print("\nTop Results:\n")

    for i, result in enumerate(results, start=1):
        print("=" * 80)
        print(f"Result {i}")
        print(f"Rank: {i}")
        print(f"Score: {result['score']:.4f}")
        print(f"Source File: {result['file']}")
        print("\nRelevant Passage:\n")
        print(result['text'][:1500])
        print()


Ask a legal question (type 'exit' to quit): exit
Exiting...


In [66]:
# ================================
# Persistent Legal Search Bar
# ================================

SIMILARITY_THRESHOLD = 0.35  # Adjust if needed

print("=" * 80)
print("Bangladesh Legal AI Search")
print("Type your legal question below.")
print("Type 'exit' to stop.")
print("=" * 80)

while True:

    user_query = input("\n🔎 Search: ").strip()

    # Exit condition
    if user_query.lower() == "exit":
        print("Goodbye!")
        break

    # Empty input handling
    if len(user_query) == 0:
        print("⚠️ Please enter a legal question.")
        continue

    # Encode the query
    query_embedding = model.encode(
        [user_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search top 5
    scores, indices = index.search(query_embedding, 5)

    best_score = float(scores[0][0])

    # If similarity is too low
    if best_score < SIMILARITY_THRESHOLD:
        print("\n❌ I could not find a sufficiently relevant legal passage.")
        print("Please try asking a clearer or more specific legal question.")
        continue

    print("\n✅ Top Matching Results:\n")

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):

        print("=" * 80)
        print(f"Rank: {rank}")
        print(f"Similarity Score: {score:.4f}")
        print(f"Source File: {df.iloc[idx]['file']}")
        print("-" * 80)

        preview = df.iloc[idx]["text"][:1200]
        print(preview)
        print()

Bangladesh Legal AI Search
Type your legal question below.
Type 'exit' to stop.

🔎 Search: What are tax laws?

✅ Top Matching Results:

Rank: 1
Similarity Score: 0.5287
Source File: act-print-44.json
--------------------------------------------------------------------------------
The Municipal Taxation Act, 1881 XI 1881 19/07/2025 1. This Act may be called theMunicipal Taxation Act, 1881. It extends to the whole of2[Bangladesh]. 2. In this Act “Municipal Committee”3[includes a Paurashava or a Municipal Corporation] or a body of Municipal Commissioners constituted by or under the provisions of any enactment for the time being in force. 3. Notwithstanding anything contained in any enactment for the time being in force, the Government may, by an order in writing, prohibit the levy by a Municipal Committee of any specified tax-(a)	payable by any person subject to4[theArmy Act, 1952, the Naval Discipline Ordinance, 1961 ortheAir Force Act, 1953]  who is compelled by the exigencies of Milita

In [58]:
!pip install -q gradio

In [ ]:
import gradio as gr

SIMILARITY_THRESHOLD = 0.35

def legal_search_app(user_query):
    user_query = user_query.strip()

    if not user_query:
        return "⚠️ Please enter a legal question."

    try:
        query_embedding = model.encode(
            [user_query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        scores, indices = index.search(query_embedding, 5)

        if len(scores[0]) == 0 or float(scores[0][0]) < SIMILARITY_THRESHOLD:
            return (
                "❌ No sufficiently relevant legal information was found.\n\n"
                "Please try asking a clearer or more specific legal question."
            )

        output = ""

        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):

            source_name = (
                df.iloc[idx]["file"]
                if "file" in df.columns
                else "Unknown Source"
            )

            text_value = (
                df.iloc[idx]["text"]
                if "text" in df.columns
                else str(df.iloc[idx])
            )

            similarity_percent = score * 100

            output += (
                f"## Result {rank}\n"
                f"**Similarity Score:** {similarity_percent:.2f}%\n"

                f"**Source:** {source_name}\n\n"
                f"{text_value[:1500]}\n\n"
                f"{'-'*60}\n\n"
            )

        return output

    except Exception as e:
        return f"⚠️ An error occurred while processing the query:\n\n{str(e)}"


demo = gr.Interface(
    fn=legal_search_app,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Example: What is the punishment for theft in Bangladesh?",
        label="🔎 Ask a Legal Question"
    ),
    outputs=gr.Markdown(label="Search Results"),
    title="Bangladesh Judiciary Legal AI Search",
    description=(
        "Ask legal questions in natural language. "
        "The system searches the indexed legal documents "
        "and returns the most relevant passages."
    ),
    allow_flagging="never"
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://81df5b32e57cc79fa1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
